In [ ]:
from io import StringIO
import csv
import pandas as pd
import math
from scipy.stats import pearsonr
from matplotlib import pyplot as plt
import matplotlib as mpl
import numpy as np
import re
import itertools
from collections import Counter
import seaborn as sns
import seaborn.objects as so
import json
import os
import multiprocessing as mp

In [ ]:
WORKDIR="../datasets/qe-ts/"
YML_FILE="experiment-1-1.yml.pkl"

In [ ]:
%%capture output
%%bash -s "$WORKDIR" "$YML_FILE"
cd $1;
./postprocess.sh ts_storage_stats $2

In [ ]:
storage_stats_csv = output.stdout
storage_stats_csv

In [ ]:
%%capture output
%%bash -s "$WORKDIR" "$YML_FILE"
cd $1;
./postprocess.sh ts_locust_stats $2

In [ ]:
locust_stats_csv = output.stdout
locust_stats_csv

In [ ]:
%%capture output
%%bash -s "$WORKDIR" "$YML_FILE"
cd $1;
./postprocess.sh print_output_paths $2

In [ ]:
output_paths_csv = output.stdout
output_paths_csv

In [ ]:
%%capture output
%%bash -s "$WORKDIR" "$YML_FILE"
cd $1;
./postprocess.sh ts_locust_history_stats $2

In [ ]:
locust_history_stats_csv = output.stdout
locust_history_stats_csv

In [ ]:
f = StringIO(storage_stats_csv)
storage_stats = pd.read_csv(f)
storage_stats

f2 = StringIO(locust_stats_csv)
locust_stats = pd.read_csv(f2)
locust_stats

f3 = StringIO(output_paths_csv)
output_paths = pd.read_csv(f3)
output_paths

f4 = StringIO(locust_history_stats_csv)
locust_history_stats = pd.read_csv(f4)
locust_history_stats

In [ ]:
# experiment11_var-zipf-16_e_8_substring_7_7
pattern_e11 = r"experiment11_(?P<Svar_or_fixed>[a-z]+)-(?P<Szipf_or_uniform>[a-z]+)-(?P<Imlen>[0-9]+)_(?P<Se_or_u>[a-z])(_(?P<Icf>[0-9]+)_(?P<Sss_or_pref>[a-z]+)_(?P<Ilb>[0-9]+)_(?P<Iub>[0-9]+))?"

for st in [locust_stats, storage_stats, locust_history_stats]:
    keys_to_add = {}
    for name in st["Task Name"]:
        d = re.match(pattern_e11, name).groupdict()
        for k, v in d.items():
            if k[0] == 'I':
                v = int(v) if v is not None else -1
            elif k[0] == 'S':
                v = str(v) if v is not None else ""
            k = k[1:]
            if k not in keys_to_add:
                keys_to_add[k] = []
            keys_to_add[k].append(v)

    for k, v in keys_to_add.items():
        st[k] = v

In [ ]:
locust_stats_name_map = {'Task Name': 'name', 'Request Count': 'count', 'Failure Count': 'failures', 'Average Response Time': 'lat_avg', 'Min Response Time': 'lat_min', 'Max Response Time': 'lat_max',
                         'Requests/s': 'req_per_s', 'Failures/s': 'fail_per_s'}
for pcnt in ['50%', '66%', '75%', '80%', '90%', '95%','98%', '99%', '99.9%', '99.99%']:
    locust_stats_name_map[pcnt] = f'lat{pcnt[:-1]}'
locust_stats_drop_cols = ['Execution', 'Type', 'Name', 'Median Response Time', 'Average Content Size', '100%']

# Filter out aggregated column
locust_stats = locust_stats[locust_stats['Name'] != 'Aggregated']

locust_stats = locust_stats.drop(columns=locust_stats_drop_cols)
locust_stats = locust_stats.rename(columns=locust_stats_name_map)

locust_stats.columns

In [ ]:
storage_stats_name_map = {'Task Name': 'name', 'Name': 'stat_name', 'Total Objects': 'count', 'Uncompressed Data Size': 'data_size', 'Compressed Data Size': 'compressed_size', 'Index Size': 'index_size', 'Total Compressed Size': 'total_size'}
storage_stats_drop_cols = ['Execution']
storage_stats = storage_stats.drop(columns=storage_stats_drop_cols)
storage_stats = storage_stats.rename(columns=storage_stats_name_map)

storage_stats

In [ ]:
#Timestamp,User Count,Type,Name,Requests/s,Failures/s,50%,66%,75%,80%,90%,95%,98%,99%,99.9%,99.99%,100%,Total Request Count,Total Failure Count,Total Median Response Time,Total Average Response Time,Total Min Response Time,Total Max Response Time,Total Average Content Size

locust_history_stats_name_map = {'Task Name': 'name', 'Total Request Count': 'count', 'Total Failure Count': 'failures', 'Total Average Response Time': 'lat_avg', 'Total Min Response Time': 'lat_min', 'Total Max Response Time': 'lat_max',
                         'Requests/s': 'req_per_s', 'Failures/s': 'fail_per_s', 'Timestamp': 'ts'}
for pcnt in ['50%', '66%', '75%', '80%', '90%', '95%','98%', '99%', '99.9%', '99.99%']:
    locust_history_stats_name_map[pcnt] = f'lat{pcnt[:-1]}'
locust_history_stats_drop_cols = ['Execution', 'Type', 'Name', 'Total Median Response Time', 'Total Average Content Size', '100%', 'User Count']

# Filter out aggregated column
locust_history_stats = locust_history_stats[locust_history_stats['Name'] != 'Aggregated']

locust_history_stats = locust_history_stats.drop(columns=locust_history_stats_drop_cols)
locust_history_stats = locust_history_stats.rename(columns=locust_history_stats_name_map)

locust_history_stats.columns

In [ ]:
output_paths = output_paths.drop(columns=['Execution'])
output_paths = output_paths.rename(columns={'Task Name': 'name', 'Output Path': 'path'})
output_paths

In [ ]:
unq_colls = storage_stats['stat_name'].unique()
merged_stats = locust_stats
for stat_name in unq_colls:
    split_stat = storage_stats[storage_stats['stat_name'] == stat_name]
    rem_cols = ['var_or_fixed', 'zipf_or_uniform', 'mlen', 'e_or_u', 'cf', 'ss_or_pref', 'lb', 'ub', 'stat_name']
    prefix = stat_name.split('_')[-1].lower()
    col_map = {c: prefix + "_" + c for c in split_stat.columns if c != 'name'}
    renamed_split = split_stat.drop(columns=rem_cols).rename(columns=col_map)
    merged_stats = pd.merge(merged_stats, renamed_split, on='name', how='inner')

print(merged_stats.columns)
merged_stats

In [ ]:
# Helper to find paths in FTDC
def recursive_find(obj, key, cur=''):
    if type(obj) != dict:
        return []
    found = []
    for k, v in obj.items():
        k_str = f'["{k}"]'
        if k == key:
            found.append((cur + k_str, v))
        found.extend(recursive_find(v, key, cur + k_str))
    return found

# Get all WC stats for a given testname, path
def process_row(itrow):
    i, row = itrow
    print(f'Processing row {i}')
    all_wc_data = pd.DataFrame(columns=['name', 'start', 'end', 'conflicts'])
    name = row['name']
    with open(os.path.join(WORKDIR, row['path'], 'metrics.json'), 'r') as f:
        for lin in f.readlines():
            try:
                obj = json.loads(lin)
            except Exception as e:
                print(f'JSON parse failed:\n{e}\n')
                return (i, row, "Parsing failure")
            wc = obj["serverStatus"]["metrics"]["operation"]["writeConflicts"]
            start_ts = obj['start']
            end_ts = obj['end']
            all_wc_data.loc[len(all_wc_data)] = [name, start_ts, end_ts, wc]
    return all_wc_data

with mp.Pool(16) as p:
    dfs = p.map(process_row, output_paths.iterrows())

In [ ]:
# set up full write conflict stats
# note: if we fail here, we have invalid json; find rows where this occurs and fix them
all_writeconflict_stats = pd.concat(dfs).reset_index()
all_writeconflict_stats

In [ ]:
# add final write conflict stats to merged_stats
name_to_cft = {}
for df in dfs:
    name = df.iloc[0]['name']
    name_to_cft[name] = df['conflicts'].max()

merged_stats['conflicts'] = merged_stats.apply(lambda row: name_to_cft[row['name']], axis=1)
merged_stats['conflicts']

In [ ]:
# useful segmentations
enc_stats = merged_stats[merged_stats['e_or_u'] == 'e']
unenc_stats = merged_stats[merged_stats['e_or_u'] == 'u']

In [ ]:
import bisect
import random


def calculate_harmonic_sum(n, alpha):
    total_sum = 0.0
    for i in range(1, n + 1):
        total_sum = total_sum + 1 / math.pow(i, alpha)
    return total_sum


def get_zipf_docs(values, n):
    sum_freq = 0
    # numValues = <SET THIS>
    alpha = 1.1
    freq_list = []
    exact_freq_list = []

    h = calculate_harmonic_sum(len(values), alpha)

    for i in range(len(values)):
        exact_f_i = n * (math.pow(i + 1, -alpha) / h)
        f_i = math.ceil(exact_f_i)
        freq_list.append(f_i)
        exact_freq_list.append(exact_f_i)
        sum_freq += f_i

    if sum_freq > n:
        assert freq_list[0] > 0
        print("off by" + str(sum_freq - n))
        off_by_after_sub = [
            abs((freq_list[i] - 1 - exact_freq_list[i]) / exact_freq_list[i])
            for i in range(len(freq_list))
        ]
        sorted_off_by = [(-offby, i) for i, offby in enumerate(off_by_after_sub)]
        sorted_off_by.sort()
        while sum_freq > n:
            _, i = sorted_off_by.pop()
            assert freq_list[i] > 0
            freq_list[i] -= 1
            sum_freq -= 1
            new_offby = abs((freq_list[i] - 1 - exact_freq_list[i]) / exact_freq_list[i])
            bisect.insort(sorted_off_by, (-new_offby, i))

        assert sum(freq_list) == n
        assert freq_list[0] > 0

    docs = []
    for i in range(len(values)):
        docs += [values[i]] * freq_list[i]

    random.shuffle(docs)
    return docs


def get_docs(coll_type, n):
    t = coll_type.split("-")
    assert len(t) == 3
    assert t[0] in ("fixed", "var")
    assert t[1] in ("uniform", "zipf")
    mlen = int(t[2])
    if t[1] == "zipf":
        values = set()
        if t[0] == "fixed":
            while len(values) < 1000:
                values.add("".join(random.choices("0123456789", k=mlen)))
        else:
            while len(values) < 1000:
                slen = random.randint(1, mlen)
                values.add("".join(random.choices("0123456789", k=slen)))
        lvalues = list(values)
        random.shuffle(lvalues)
        return get_zipf_docs(lvalues, n)
    else:
        if t[0] == "fixed":
            return ["".join(random.choices("0123456789", k=mlen)) for _ in range(n)]
        else:
            return [
                "".join(random.choices("0123456789", k=random.randint(1, mlen))) for _ in range(n)
            ]

In [ ]:
colltypes = ['fixed-uniform-16', 'var-uniform-16', 'fixed-zipf-16', 'var-zipf-16', 'fixed-uniform-32', 'var-uniform-32']
docsets = {k: get_docs(k, 100000) for k in colltypes}
mydata = pd.DataFrame(columns=["colltype", "itype", "lb", "ub", "nreal", "ndummy", "ntotal"])
for typ in colltypes:
    for idx in ('prefix', 'substring'):
        for lb in (1, 3, 5, 7):
            if idx == 'substring' and lb == 1:
                continue
            for ub in range(lb, 8, 2):
                nreal = 0
                ndummy = 0
                for doc in docsets[typ]:
                    real_len = len(doc)
                    mlen = int(typ.split('-')[2])
                    cbc_len = min(mlen, math.ceil((real_len + 5) / 16.) * 16 - 5)
                    if idx == 'prefix':
                        smallest_tag = lb
                        largest_tag = min(ub, real_len)
                        real_tags = 1 + max(largest_tag - smallest_tag + 1, 0)
                        total_tags = 2 + ub - lb
                        nreal += real_tags
                        ndummy += total_tags - real_tags
                    else:
                        real_tags = set()
                        mclen = min(cbc_len, mlen)
                        total_tags = 0
                        for taglen in range(lb, ub+1):
                            if taglen > mclen:
                                break
                            total_tags += mclen - taglen + 1
                            if taglen > real_len:
                                continue
                            for i in range(0, len(doc) - taglen + 1):
                                real_tags.add(doc[i:i+taglen])
                        
                        nreal += 1 + len(real_tags)
                        ndummy += total_tags - len(real_tags)
                mydata.loc[len(mydata)] = [typ, idx, lb, ub, nreal, ndummy, nreal + ndummy]
                print(f'{typ}, {idx}, ({lb}, {ub}): real: {nreal}, dummy: {ndummy}, total: {nreal + ndummy}')
mydata


In [ ]:
sns.set_style("whitegrid")

In [ ]:
# Graph 1
for typ in ('prefix', 'substring'):
    stats = enc_stats[(enc_stats['ss_or_pref'] == typ) & (enc_stats['var_or_fixed'] == 'fixed') & (enc_stats['zipf_or_uniform'] == 'uniform') & (enc_stats['cf'] == 4)].copy()
    stats['window_size'] = stats['ub'] - stats['lb'] + 1
    # for row in stats.rows():
    #     row['lbub'] = (row['lb'], row['ub'])
    # stats.groupby('window_size')
    # print(stats[['name', 'window_size', 'lb', 'edc_data_size', 'esc_count']])
    stats['lb_aba'] = stats['lb'].astype(str)
    stats['edc_data_size_mb'] = stats['edc_data_size'] / 1000000.
    title = f'Uncompressed EDC size vs window size by lb, {typ}'
    plt.close()
    ax = plt.gca()
    p = so.Plot(stats, x='window_size', y='edc_data_size_mb', color='lb')
    p = p.add(so.Bar(alpha=1), so.Dodge(empty='drop'))
    p = p.label(title=title, x='Window size', y='EDC size (MB)')
    p = p.theme(sns.axes_style(style='whitegrid'))
    p = p.scale(x=so.Continuous().tick(at=stats['window_size'].unique()), color=so.Continuous().tick(at=stats['lb'].unique()))
    p = p.layout(engine='tight')
    p = p.on(ax)
    sns.despine(ax=ax, left=True)
    ax.grid(False, axis='x')

    p.save(f'../figures/graph1-{typ}.png', bbox_inches='tight')
    display(p)
    # g = sns.catplot(data=stats, kind='bar', x='window_size', y='edc_data_size_mb', col='lb', sharex=False, hue='lb')
    # g.set(title=title, xlabel='Window size', ylabel='EDC size (MB)')
    # g.despine(left=True)


In [ ]:
# Graph 2
for typ in ('prefix', 'substring'):
    lowest_lb = 1 if typ == 'prefix' else 3
    stats = enc_stats[(enc_stats['ss_or_pref'] == typ) & (enc_stats['cf'] == 8) & (enc_stats['lb'] == lowest_lb)].copy()

    title = f'Tags per document vs (lb, ub) by collection, {typ}'
    stats['avg_tag_count'] = stats['esc_count'] / stats['edc_count']
    stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
    print(stats[['ub', 'avg_tag_count', 'collname']])
    if typ == 'prefix':
        stats = stats[stats['collname'] == 'fixed, uniform']
        stats['collname'] = '*'
    else:
        stats = stats[stats['collname'] != 'fixed, uniform']
        stats['collname'] = stats['collname'].replace('fixed, zipf', 'fixed, *')
    plt.close()
    ax = plt.gca()
    p = so.Plot(stats, x='ub', y='avg_tag_count', color='collname')
    p = p.add(so.Line(alpha=1, marker='o'))
    p = p.label(title=title, x='(lb, ub)', y='Average tags per EDC document', color='Collection')
    p = p.theme(sns.axes_style(style='whitegrid'))
    p = p.layout(engine='tight')
    p = p.scale(x=so.Continuous().tick(at=stats['ub'].unique()).label(like=f'({lowest_lb}, {{x}})'))
    p = p.on(ax)
    sns.despine(ax=ax, left=True, bottom=True)
    # ax.grid(False, axis='x')

    p.save(f'../figures/graph2-{typ}.png', bbox_inches='tight')
    display(p)
    plt.close()
    
    # g = sns.catplot(data=stats, kind='bar', x='window_size', y='edc_data_size_mb', col='lb', sharex=False, hue='lb')
    # g.set(title=title, xlabel='Window size', ylabel='EDC size (MB)')
    # g.despine(left=True)


In [ ]:
# Graph 3
for typ in ('prefix', 'substring'):
    lowest_lb = 1 if typ == 'prefix' else 3
    stats = enc_stats[(enc_stats['ss_or_pref'] == typ) & (enc_stats['cf'] == 8) & (enc_stats['lb'] == lowest_lb)].copy()

    
    stats['avg_tag_count'] = stats['esc_count'] / stats['edc_count']
    stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
    print(stats[['ub', 'avg_tag_count', 'collname']])
    stackstat = None
    for coll in ('edc', 'esc', 'ecoc'):
        cpy = stats.copy()
        cpy['whichcoll'] = [coll] * len(cpy)
        cpy['coll_data_size'] = cpy[coll + '_data_size'] / 1000000
        if stackstat is None:
            stackstat = cpy
        else:
            stackstat = pd.concat([stackstat, cpy])

    stackstat = stackstat.reset_index()
    
    # print(stackstat)

    for collname in stackstat['collname'].unique():
        title = f'Collection size vs (lb, ub) by subcollection, {typ}, {collname}'
        print(collname)
        substat = stackstat[stackstat['collname'] == collname]
        print(substat)
        plt.close()
        ax = plt.gca()
        p = so.Plot(data=substat, x='ub', y='coll_data_size', color='whichcoll')
        p = p.add(so.Bar(alpha=1), so.Dodge())
        p = p.label(title=title, x='(lb, ub)', y='Collection size (MB)', color='Collection')
        p = p.theme(sns.axes_style(style='whitegrid'))
        p = p.layout(engine='tight')
        p = p.scale(x=so.Continuous().tick(at=stats['ub'].unique()).label(like=f'({lowest_lb}, {{x}})'))
        p = p.on(ax)
        sns.despine(ax=ax, left=True, bottom=True)
        ax.grid(False, axis='x')

        p.save(f"../figures/graph3-{typ}-{collname.split(', ')[0]}-{collname.split(', ')[1]}.png", bbox_inches='tight')
        display(p)
        plt.close()
    
    # g = sns.catplot(data=stats, kind='bar', x='window_size', y='edc_data_size_mb', col='lb', sharex=False, hue='lb')
    # g.set(title=title, xlabel='Window size', ylabel='EDC size (MB)')
    # g.despine(left=True)

In [ ]:
# Graph 4
for typ in ('prefix', 'substring'):
    lowest_lb = 1 if typ == 'prefix' else 3
    stats = mydata[(mydata['itype'] == typ) & (mydata['lb'] == lowest_lb) & (mydata['colltype'].map(lambda c: c.endswith('-16')))].copy()

    title = f'Proportion of dummy tags vs (lb, ub) by collection, {typ}'
    stats['collname'] = stats['colltype'].map(lambda s: s[:-3].replace('-', ', '))
    stats['ratio'] = stats['ndummy'] / stats['ntotal']
    if typ == 'prefix':
        stats = stats[stats['collname'] != 'fixed, uniform']
        stats['collname'] = stats['collname'].replace('fixed, zipf', 'fixed, *')
    plt.close()
    ax = plt.gca()
    p = so.Plot(stats, x='ub', y='ratio', color='collname')
    p = p.add(so.Line(alpha=1, marker='o'))
    p = p.label(title=title, x='(lb, ub)', y='(Dummy tags) / (Total tags)', color='Collection')
    p = p.theme(sns.axes_style(style='whitegrid'))
    p = p.layout(engine='tight')
    p = p.scale(x=so.Continuous().tick(at=stats['ub'].unique()).label(like=f'({lowest_lb}, {{x}})'))
    p = p.on(ax)
    sns.despine(ax=ax, left=True, bottom=True)
    # ax.grid(False, axis='x')

    p.save(f'../figures/graph4-{typ}.png', bbox_inches='tight')
    display(p)
    plt.close()
    

In [ ]:
# Graph 5
for typ in ('prefix', 'substring'):
    lowest_lb = 1 if typ == 'prefix' else 3
    stats = enc_stats[(enc_stats['ss_or_pref'] == typ) & (enc_stats['cf'] == 8) & (enc_stats['lb'] == lowest_lb)].copy()

    title = f'Ratio of encrypted to unencrypted size vs (lb, ub) by collection, {typ}'
    stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
    sub_unenc_stats = unenc_stats.copy()
    sub_unenc_stats['collname'] = sub_unenc_stats['var_or_fixed'] + ', ' + sub_unenc_stats['zipf_or_uniform']
    print(sub_unenc_stats['edc_total_size'])
    print(stats['edc_total_size'])
    stats['ratio'] = stats.apply(lambda row: (row['edc_total_size'] + row['esc_total_size'] + row['ecoc_total_size']) / sub_unenc_stats[sub_unenc_stats['collname'] == row['collname']]['edc_total_size'].mean(), axis=1)
    # print(stats[['ub', 'avg_tag_count', 'collname']])
    # if typ == 'prefix':
    #     stats = stats[stats['collname'] == 'fixed, uniform']
    #     stats['collname'] = '*'
    # else:
    #     stats = stats[stats['collname'] != 'fixed, uniform']
    #     stats['collname'] = stats['collname'].replace('fixed, zipf', 'fixed, *')
    plt.close()
    ax = plt.gca()
    p = so.Plot(stats, x='ub', y='ratio', color='collname')
    p = p.add(so.Line(alpha=1, marker='o'))
    p = p.label(title=title, x='(lb, ub)', y='(Total size) / (Total unencrypted size)', color='Collection')
    p = p.theme(sns.axes_style(style='whitegrid'))
    p = p.layout(engine='tight')
    p = p.scale(x=so.Continuous().tick(at=stats['ub'].unique()).label(like=f'({lowest_lb}, {{x}})'))
    p = p.on(ax)
    sns.despine(ax=ax, left=True, bottom=True)
    # ax.grid(False, axis='x')

    p.save(f'../figures/graph5-{typ}.png', bbox_inches='tight')
    display(p)
    plt.close()
    
    # g = sns.catplot(data=stats, kind='bar', x='window_size', y='edc_data_size_mb', col='lb', sharex=False, hue='lb')
    # g.set(title=title, xlabel='Window size', ylabel='EDC size (MB)')
    # g.despine(left=True)


In [ ]:
# Graph 6
for typ in ('prefix', 'substring'):
    lowest_lb = 1 if typ == 'prefix' else 3
    stats = mydata[(mydata['itype'] == typ) & (mydata['lb'] == lowest_lb) & (mydata['colltype'].map(lambda s: s.endswith('-32')))].copy()
    print(stats)
    title = f'Proportion of dummy tags vs (lb, ub) by collection, {typ}, maxlen=32'
    stats['collname'] = stats['colltype'].map(lambda s: s[:-3].replace('-', ', '))
    stats['ratio'] = stats['ndummy'] / stats['ntotal']
    # if typ == 'prefix':
    #     stats = stats[stats['collname'] != 'fixed, uniform']
    #     stats['collname'] = stats['collname'].replace('fixed, zipf', 'fixed, *')
    plt.close()
    ax = plt.gca()
    p = so.Plot(stats, x='ub', y='ratio', color='collname')
    p = p.add(so.Line(alpha=1, marker='o'))
    p = p.label(title=title, x='(lb, ub)', y='(Dummy tags) / (Total tags)', color='Collection')
    p = p.theme(sns.axes_style(style='whitegrid'))
    p = p.layout(engine='tight')
    p = p.scale(x=so.Continuous().tick(at=stats['ub'].unique()).label(like=f'({lowest_lb}, {{x}})'))
    p = p.on(ax)
    sns.despine(ax=ax, left=True, bottom=True)
    # ax.grid(False, axis='x')

    p.save(f'../figures/graph6-{typ}.png', bbox_inches='tight')
    display(p)
    plt.close()
    

In [ ]:
# Graph ME: cf + colltype vs storage size
for typ in ('prefix', 'substring'):
    stats = enc_stats[(enc_stats['ss_or_pref'] == typ) & (enc_stats['lb'] == 3) & (enc_stats['mlen'] == 16)].copy()
    stats['window_size'] = stats['ub'] - stats['lb'] + 1
    # for row in stats.rows():
    #     row['lbub'] = (row['lb'], row['ub'])
    # stats.groupby('window_size')
    # print(stats[['name', 'window_size', 'lb', 'edc_data_size', 'esc_count']])
    stats['lb_aba'] = stats['lb'].astype(str)
    stats['edc_data_size_mb'] = stats['edc_data_size'] / 1000000.
    stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
    print(stats[['collname', 'cf', 'edc_data_size_mb', 'edc_count']])
    title = f'Uncompressed EDC size vs collection by cf, {typ}'
    plt.close()
    ax = plt.gca()
    p = so.Plot(stats, x='collname', y='edc_data_size_mb', color='cf')
    p = p.add(so.Bar(alpha=1), so.Dodge(empty='drop'))
    p = p.label(title=title, x='Collection', y='EDC size (MB)')
    p = p.theme(sns.axes_style(style='whitegrid'))
    p = p.scale(color=so.Continuous().tick(at=sorted(stats['cf'].unique())))
    p = p.layout(engine='tight')
    p = p.on(ax)
    sns.despine(ax=ax, left=True)
    ax.grid(False, axis='x')

    p.save(f'../figures/graph7-{typ}.png', bbox_inches='tight')
    display(p)
    # g = sns.catplot(data=stats, kind='bar', x='window_size', y='edc_data_size_mb', col='lb', sharex=False, hue='lb')
    # g.set(title=title, xlabel='Window size', ylabel='EDC size (MB)')
    # g.despine(left=True)


In [ ]:
# Graph i1
for typ in ('prefix', 'substring'):
    for cf in (0, 4, 8, 16, 32):
        stats = enc_stats[(enc_stats['ss_or_pref'] == typ) & (enc_stats['cf'] == cf) & (enc_stats['ub'] == 7)].copy()
        stats['tp'] = 16. * 1e9 / stats['lat_avg']
        stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
        title = f'Throughput vs (lb, ub) by collection, {typ}, cf={cf}'
        plt.close()
        ax = plt.gca()
        p = so.Plot(stats, x='lb', y='tp', color='collname')
        p = p.add(so.Bar(alpha=1), so.Dodge(empty='drop'))
        p = p.label(title=title, x='(lb, ub)', y='Throughput (ops/sec)', color='Collection')
        p = p.theme(sns.axes_style(style='whitegrid'))
        p = p.layout(engine='tight')
        p = p.scale(x=so.Continuous().tick(at=stats['lb'].unique()).label(like='({x}, 7)'))
        p = p.on(ax)
        sns.despine(ax=ax, left=True)
        ax.grid(False, axis='x')

        p.save(f'../figures/graphi1-{typ}-{cf}.png', bbox_inches='tight')
        display(p)


In [ ]:
# uncompressed vs compressed vs total investigation
stats = enc_stats.copy()
stats['esc_c_ratio'] = stats['esc_compressed_size'] / stats['esc_data_size']
stats['edc_c_ratio'] = stats['edc_compressed_size'] / stats['edc_data_size']
stats['ecoc_c_ratio'] = stats['ecoc_compressed_size'] / stats['ecoc_data_size']
stats['total_c_ratio'] = (
    stats['esc_compressed_size'] + stats['edc_compressed_size'] + stats['ecoc_compressed_size']
) / (
    stats['esc_data_size'] + stats['edc_data_size'] + stats['ecoc_data_size']
)

esc_sort = stats.sort_values(by='esc_c_ratio')


edc_sort = stats.sort_values(by='edc_c_ratio')


ecoc_sort = stats.sort_values(by='ecoc_c_ratio')
# ecoc_sort[['ecoc_c_ratio', 'name']], edc_sort[['edc_c_ratio', 'name']], esc_sort[['esc_c_ratio', 'name']], len(stats[stats['edc_c_ratio'] > 1]), len(stats[stats['total_c_ratio'] > 1])
esc_sort[['esc_c_ratio', 'name']]


In [ ]:
# Graph i2
for typ in ('prefix', 'substring'):
    for cf in (0, 4, 8, 16, 32):
        lowest_lb = 1 if typ == 'prefix' else 3
        stats = enc_stats[(enc_stats['ss_or_pref'] == typ) & (enc_stats['cf'] == cf) & (enc_stats['lb'] == lowest_lb)].copy()
        stats['tp'] = 16. * 1e9 / stats['lat_avg']
        stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
        title = f'Throughput vs (lb, ub) by collection, {typ}, cf={cf}'
        plt.close()
        ax = plt.gca()
        p = so.Plot(stats, x='ub', y='tp', color='collname')
        p = p.add(so.Line(marker='o', alpha=1))
        p = p.label(title=title, x='(lb, ub)', y='Throughput (ops/sec)', color='Collection')
        p = p.theme(sns.axes_style(style='whitegrid'))
        p = p.layout(engine='tight')
        p = p.scale(x=so.Continuous().tick(at=stats['ub'].unique()).label(like=f'({lowest_lb}, {{x}})'))
        p = p.on(ax)
        sns.despine(ax=ax, left=True)

        p.save(f'../figures/graphi2-{typ}-{cf}.png', bbox_inches='tight')
        display(p)


In [ ]:
# Graph i3
for typ in ('prefix', 'substring'):
    for collname in (enc_stats['var_or_fixed'] + ', ' + enc_stats['zipf_or_uniform']).unique():
        for cf in (0, 4, 8, 16, 32):
            lowest_lb = 1 if typ == 'prefix' else 3
            stats = enc_stats.copy()
            stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
            stats = stats[(stats['ss_or_pref'] == typ) & (stats['cf'] == cf) & (stats['collname'] == collname)].copy()
            stats['tp'] = 16. * 1e9 / stats['lat_avg']
            
            title = f'Throughput vs ub by lb, {typ}, cf={cf}, {collname}'
            plt.close()
            ax = plt.gca()
            p = so.Plot(stats, x='ub', y='tp', color='lb')
            p = p.add(so.Bar(alpha=1), so.Dodge(empty='drop'))
            p = p.label(title=title, x='ub', y='Throughput (ops/sec)', color='lb')
            p = p.theme(sns.axes_style(style='whitegrid'))
            p = p.layout(engine='tight')
            p = p.scale(x=so.Continuous().tick(at=stats['ub'].unique()), color=so.Continuous().tick(at=stats['lb'].unique()))
            p = p.on(ax)
            sns.despine(ax=ax, left=True)
            ax.grid(False, axis='x')

            p.save(f'../figures/graphi3-{typ}-{cf}-{collname.split(", ")[0]}-{collname.split(", ")[1]}.png', bbox_inches='tight')
            display(p)


In [ ]:
# Graph i4
for typ in ('prefix', 'substring'):
    for u_or_z in enc_stats['zipf_or_uniform'].unique():
        lowest_lb = 1 if typ == 'prefix' else 3
        stats = enc_stats[(enc_stats['ss_or_pref'] == typ) & (enc_stats['zipf_or_uniform'] == u_or_z) & (enc_stats['var_or_fixed'] == 'fixed') & (enc_stats['ub'] == 7)].copy()
        stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
        stats['tp'] = 16. * 1e9 / stats['lat_avg']
        
        title = f'Throughput vs contention factor by (lb, ub), {typ}, fixed, {u_or_z}'
        plt.close()
        ax = plt.gca()
        p = so.Plot(stats, x='cf', y='tp', color='lb')
        p = p.add(so.Line(marker='o', alpha=1))
        p = p.label(title=title, x='Contention factor', y='Throughput (ops/sec)', color='(lb, ub)')
        p = p.theme(sns.axes_style(style='whitegrid'))
        p = p.layout(engine='tight')
        p = p.scale(x=so.Continuous().tick(at=stats['cf'].unique()), color=so.Continuous().tick(at=stats['lb'].unique()).label(like='({x}, 7)'))
        p = p.on(ax)
        sns.despine(ax=ax, left=True)

        p.save(f'../figures/graphi4-{typ}-{u_or_z}.png', bbox_inches='tight')
        display(p)


In [ ]:
# Graph i5
for typ in ('prefix', 'substring'):
    for u_or_z in enc_stats['zipf_or_uniform'].unique():
        lowest_lb = 1 if typ == 'prefix' else 3
        stats = enc_stats[(enc_stats['ss_or_pref'] == typ) & (enc_stats['zipf_or_uniform'] == u_or_z) & (enc_stats['var_or_fixed'] == 'fixed') & (enc_stats['ub'] - enc_stats['lb'] == 2)].copy()
        stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
        stats['tp'] = 16. * 1e9 / stats['lat_avg']
        
        title = f'Throughput vs contention factor by (lb, ub), {typ}, fixed, {u_or_z}'
        plt.close()
        ax = plt.gca()
        p = so.Plot(stats, x='cf', y='tp', color='lb')
        p = p.add(so.Line(marker='o', alpha=1))
        p = p.label(title=title, x='Contention factor', y='Throughput (ops/sec)', color='(lb, ub)')
        p = p.theme(sns.axes_style(style='whitegrid'))
        p = p.layout(engine='tight')
        p = p.scale(x=so.Continuous().tick(at=stats['cf'].unique()), color=so.Continuous().tick(at=stats['lb'].unique()).label(like=lambda x, _: f'({x}, {x + 2})'))
        p = p.on(ax)
        sns.despine(ax=ax, left=True)

        p.save(f'../figures/graphi5-{typ}-{u_or_z}.png', bbox_inches='tight')
        display(p)


In [ ]:
# Conflict probability investigation
docs = docsets['fixed-zipf-16']
unq_docs = set(docs)
assert len(unq_docs) == 1000
from collections import Counter
c = Counter()
c.update(docs)
print(c)

def shares_ss(a, b, l):
    a_subs = {a[i:i+l] for i in range(len(a) - l + 1)}
    b_subs = {b[i:i+l] for i in range(len(b) - l + 1)}
    return len(a_subs.intersection(b_subs)) > 0


rn = list(range(100000))
zipf_ov = zipf_miss = fixed_ov = fixed_miss = 0
for _ in range(1000000):
    i, j = random.sample(rn, 2)
    if shares_ss(docsets['fixed-zipf-16'][i], docsets['fixed-zipf-16'][j],2):
        zipf_ov += 1
    else:
        zipf_miss += 1
    if shares_ss(docsets['fixed-uniform-16'][i], docsets['fixed-uniform-16'][j], 2):
        fixed_ov += 1
    else:
        fixed_miss += 1

print(zipf_ov, zipf_miss, fixed_ov, fixed_miss)


In [ ]:
# Graph i ME
for typ in ('prefix', 'substring'):
    for collname in (enc_stats['var_or_fixed'] + ', ' + enc_stats['zipf_or_uniform']).unique():
        for cf in (0, 4, 8, 16, 32):
            lowest_lb = 1 if typ == 'prefix' else 3
            stats = enc_stats[(enc_stats['mlen'] == 16) & (enc_stats['ss_or_pref'] == typ) & (enc_stats['cf'] == cf) & ((enc_stats['var_or_fixed'] + ', ' + enc_stats['zipf_or_uniform']) == collname)].copy()
            stats['tp'] = 16. * 1e9 / stats['lat_avg']
            stats['collname'] = stats['var_or_fixed'] + ', ' + stats['zipf_or_uniform']
            stats['window_size'] = stats['ub'] - stats['lb'] + 1
            stats = stats[stats['window_size'] <= 3]
            title = f'Throughput vs lb by window size, {typ}, cf={cf}, {collname}'
            plt.close()
            ax = plt.gca()
            p = so.Plot(stats, x='lb', y='tp', color='window_size')
            p = p.add(so.Line(marker='o', alpha=1))
            p = p.label(title=title, x='lb', y='Throughput (ops/sec)', color='Window size')
            p = p.theme(sns.axes_style(style='whitegrid'))
            p = p.layout(engine='tight')
            p = p.scale(x=so.Continuous().tick(at=stats['lb'].unique()), color=so.Continuous().tick(at=stats['window_size'].unique()))
            p = p.on(ax)
            sns.despine(ax=ax, left=True)

            p.save(f'../figures/graphi8-{typ}-{cf}-{collname.split(", ")[0]}-{collname.split(", ")[1]}.png', bbox_inches='tight')
            display(p)


In [ ]:
# graph agg write conflicts for a random test
def trim_and_normalize_wc_data(data):
    last_0 = max(data[data['conflicts'] == 0]['start'])
    first_max = min(data[data['conflicts'] == data['conflicts'].max()]['start'])
    subdata = data[(last_0 <= data['start']) & (data['start'] <= first_max)].copy()
    subdata['start'] -= last_0
    subdata['end'] -= last_0
    return subdata

def get_runtime(row):
    name = row['name']
    substats = trim_and_normalize_wc_data(all_writeconflict_stats[(all_writeconflict_stats['name'] == name)])
    return substats['start'].max()

def fixup_hist_data(data):
    last_0 = max(data[data['count'] == data['count'].min()]['ts'])
    first_max = min(data[data['count'] == data['count'].max()]['ts'])
    subdata = data[(last_0 <= data['ts']) & (data['ts'] <= first_max)].copy()
    subdata['ts'] -= last_0
    return subdata.drop_duplicates('count', keep='first')

name = 'experiment11_var-uniform-16_e_4_prefix_1_7'
row = enc_stats[enc_stats['name'] == name].iloc[0]
print(get_runtime(row))

substats = trim_and_normalize_wc_data(all_writeconflict_stats[(all_writeconflict_stats['name'] == name)])
subhist = fixup_hist_data(locust_history_stats[(locust_history_stats['name'] == name)])
subhist['tsms'] = subhist['ts'] * 1000

plt.close()
ax = sns.lineplot(data=substats, x='start', y='conflicts', legend=False, label='Conflicts', alpha=1)

ax2 = plt.twinx()
sns.lineplot(data=subhist, x='tsms', y='count', color='red', legend=False, label='Documents inserted', alpha=1, ax=ax2)

ax = plt.gca()
ax.figure.legend()

